# 02 — Five-Class TF-IDF + Logistic Regression

## Objective

Build a classical NLP classifier for a focused five-class task: religion, age, ethnicity, gender, and non-bullying content. This is a narrower scope chosen for the moderation use case; it is not a claim that the excluded `other_cyberbullying` category is harmless or invalid.

This notebook uses the same source filtering and 70/15/15 split as Notebook 01. It filters `other_cyberbullying` **after the shared split**, so its test examples are a subset of the same held-out six-class test set.

## Notebook roadmap

1. Load data and confirm the shared split fingerprint
2. Filter each split to the five-class scope
3. Apply classical preprocessing
4. Fit TF-IDF on training texts only
5. Compare Naive Bayes and tuned Logistic Regression
6. Evaluate on held-out data
7. Inspect confusion errors and model coefficients
8. Run a matched six-class Logistic Regression control on the same retained test rows

## Environment and data

Install `requirements.txt` from the repository root. Place `cyberbullying_tweets.csv` in `data/` or `/content/` in Colab. The dataset is intentionally not included in the repository.

In [ ]:
from pathlib import Path
import hashlib
import random
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

# `contractions` is listed in requirements.txt. Keep a small fallback so the
# notebook remains usable if a fresh environment has not installed it yet.
try:
    from contractions import fix as fix_contractions
except ImportError:
    def fix_contractions(text):
        text = str(text)
        replacements = [
            (r"\bcan't\b", "cannot"), (r"\bwon't\b", "will not"),
            (r"\bshan't\b", "shall not"), (r"\bain't\b", "is not"),
            (r"n['’]t\b", " not"), (r"['’]re\b", " are"),
            (r"['’]ve\b", " have"), (r"['’]ll\b", " will"),
            (r"['’]d\b", " would"), (r"['’]m\b", " am"),
        ]
        for pattern, replacement in replacements:
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        return text

SEED = 2042
ALL_LABELS = [
    "religion", "age", "ethnicity", "gender",
    "not_cyberbullying", "other_cyberbullying"
]
RETAINED_LABELS = [
    "religion", "age", "ethnicity", "gender", "not_cyberbullying"
]
OTHER_LABEL = "other_cyberbullying"

random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 140)

print("Random seed:", SEED)

## 1. Load data and reproduce the common split

The following cell is intentionally identical to Notebook 01. Compare the printed split fingerprint across notebooks; it should match when the CSV file is the same.

In [ ]:
# Locate the dataset whether this notebook is run from the repository root,
# the notebooks/ folder, or Google Colab after uploading the CSV.
CANDIDATE_PATHS = [
    Path("data/cyberbullying_tweets.csv"),
    Path("../data/cyberbullying_tweets.csv"),
    Path("cyberbullying_tweets.csv"),
    Path("/content/cyberbullying_tweets.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find cyberbullying_tweets.csv. Place it at data/cyberbullying_tweets.csv "
        "or upload it to /content/cyberbullying_tweets.csv in Colab."
    )

raw = pd.read_csv(DATA_PATH).rename(columns={
    "tweet_text": "text",
    "cyberbullying_type": "label",
})
assert {"text", "label"}.issubset(raw.columns), (
    "Expected columns text/label or tweet_text/cyberbullying_type."
)

raw = raw.dropna(subset=["text", "label"]).copy()
raw["text"] = raw["text"].astype(str).str.strip()
raw["label"] = raw["label"].astype(str).str.strip().str.lower()
raw["label"] = raw["label"].replace({
    "not bullying": "not_cyberbullying",
    "not cyberbullying": "not_cyberbullying",
    "other bullying": "other_cyberbullying",
})
raw = raw[raw["text"].ne("") & raw["label"].isin(ALL_LABELS)].copy()

print(f"Dataset path: {DATA_PATH}")
print(f"Rows after basic missing/label checks: {len(raw):,}")
print(f"Raw exact duplicate rows: {raw.duplicated(subset=['text', 'label']).sum():,}")

# Conservative, shared deduplication key. This is deliberately separate from
# either model's actual text-cleaning function. It reduces the chance that
# punctuation/URL/mention variants of the same tweet cross data partitions.
def make_split_key(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"@\w+", " MENTIONTOKEN ", text)
    text = re.sub(r"#([\w-]+)", r"\1", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split()).strip()

raw["split_key"] = raw["text"].map(make_split_key)
raw = raw[raw["split_key"].ne("")].copy()

# If the same normalized key carries more than one original label, preserve
# those rows for inspection but exclude that key from model evaluation.
label_count_per_key = raw.groupby("split_key")["label"].nunique()
conflicting_keys = set(label_count_per_key[label_count_per_key > 1].index)
label_conflicts = raw[raw["split_key"].isin(conflicting_keys)].copy()

model_source = raw[~raw["split_key"].isin(conflicting_keys)].copy()
rows_before_dedup = len(model_source)
model_df = model_source.drop_duplicates(subset="split_key", keep="first").copy()
model_df = model_df.reset_index(drop=True)
model_df["row_id"] = np.arange(len(model_df), dtype=np.int64)

print(f"Normalized text keys with conflicting labels: {len(conflicting_keys):,}")
print(f"Rows held aside due to conflicting labels: {len(label_conflicts):,}")
print(f"Repeated normalized keys removed: {rows_before_dedup - len(model_df):,}")
print(f"Rows available for modelling: {len(model_df):,}")

if not label_conflicts.empty:
    print("Examples of conflicting normalized keys (inspect rather than assume annotation error):")
    display(label_conflicts[["text", "label", "split_key"]].head(10))

observed_labels = set(model_df["label"].unique())
assert observed_labels == set(ALL_LABELS), (
    f"Expected all six labels {ALL_LABELS}, found {sorted(observed_labels)}"
)

# One shared, deterministic 70/15/15 split from the ORIGINAL six-class data.
# Filter out 'other_cyberbullying' only after these partitions are created.
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=SEED,
    stratify=model_df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

split_parts = {"train": train_df, "validation": val_df, "test": test_df}
print(f"Train: {len(train_df):,} ({len(train_df)/len(model_df):.1%})")
print(f"Validation: {len(val_df):,} ({len(val_df)/len(model_df):.1%})")
print(f"Test: {len(test_df):,} ({len(test_df)/len(model_df):.1%})")

split_counts = pd.DataFrame({
    name: part["label"].value_counts()
    for name, part in split_parts.items()
}).fillna(0).astype(int).reindex(ALL_LABELS)
display(split_counts)

# This fingerprint should match across all three notebooks when using the same CSV.
fp_payload = "|".join(
    f"{name}:{','.join(map(str, sorted(part['row_id'].tolist())))}"
    for name, part in split_parts.items()
)
split_fingerprint = hashlib.sha256(fp_payload.encode("utf-8")).hexdigest()[:12]
print("Shared split fingerprint:", split_fingerprint)
print("Record this fingerprint when comparing results from the three notebooks.")

## 2. Filter to the five-class scope

Only after generating the original six-class train/validation/test partitions do we remove `other_cyberbullying`. This preserves a common held-out retained-class test population for the controlled comparison below.

In [ ]:
train_5 = train_df[train_df["label"].isin(RETAINED_LABELS)].copy()
val_5 = val_df[val_df["label"].isin(RETAINED_LABELS)].copy()
test_5 = test_df[test_df["label"].isin(RETAINED_LABELS)].copy()

assert set(train_5["label"].unique()) == set(RETAINED_LABELS)
assert set(val_5["label"].unique()) == set(RETAINED_LABELS)
assert set(test_5["label"].unique()) == set(RETAINED_LABELS)

print("Five-class training rows:", len(train_5))
print("Five-class validation rows:", len(val_5))
print("Five-class test rows:", len(test_5))
print("Same held-out row IDs as the retained subset of the six-class test split:",
      set(test_5["row_id"]) == set(test_df.loc[test_df["label"].isin(RETAINED_LABELS), "row_id"]))
display(pd.DataFrame({
    "train": train_5["label"].value_counts().reindex(RETAINED_LABELS),
    "validation": val_5["label"].value_counts().reindex(RETAINED_LABELS),
    "test": test_5["label"].value_counts().reindex(RETAINED_LABELS),
}).fillna(0).astype(int))

## 3. Preprocessing

Use the same classical preprocessing as the six-class TF-IDF control. Negation words (`no`, `nor`, and `not`) are preserved.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import warnings

# Use the NLTK lists when available; fall back to scikit-learn's built-in list
# if the corpus download is unavailable in a restricted environment.
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    warnings.warn("NLTK stopwords corpus unavailable; using scikit-learn stopwords.")
    STOP_WORDS = set(ENGLISH_STOP_WORDS)

# Preserve negation: removing 'not' can reverse the meaning of a tweet.
STOP_WORDS -= {"no", "nor", "not"}
LEMMATIZER = WordNetLemmatizer()
try:
    LEMMATIZER.lemmatize("validation")
    WORDNET_AVAILABLE = True
except LookupError:
    WORDNET_AVAILABLE = False
    warnings.warn("WordNet corpus unavailable; lemmatization will be skipped.")

def clean_tweet_classical(text):
    """Lightly normalize a tweet for a TF-IDF model; preserve negation words."""
    text = "" if pd.isna(text) else str(text)
    text = fix_contractions(text)
    text = re.sub(r"\r|\n", " ", text.lower())
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"@\w+", " MENTIONTOKEN ", text)
    text = re.sub(r"#([\w-]+)", r"\1", text)
    text = re.sub(r"[^\x00-\x7F]", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)

    tokens = [w for w in text.split() if w not in STOP_WORDS and len(w) >= 2]
    tokens = [LEMMATIZER.lemmatize(w) for w in tokens] if WORDNET_AVAILABLE else tokens
    text = " ".join(tokens)
    text = re.sub(r"\b(\w+)((\w)\3{2,})(\w*)\b", r"\1\3\4", text)
    text = " ".join(text.split()).strip()
    return text if text else "emptytext"

print("Classical preprocessing ready. Negations 'no', 'nor', 'not' are preserved.")

In [ ]:
X_train_text_5 = train_5["text"].map(clean_tweet_classical).tolist()
X_val_text_5 = val_5["text"].map(clean_tweet_classical).tolist()
X_test_text_5 = test_5["text"].map(clean_tweet_classical).tolist()
y_train_5 = train_5["label"].to_numpy()
y_val_5 = val_5["label"].to_numpy()
y_test_5 = test_5["label"].to_numpy()

print("Example before cleaning:")
print(train_5["text"].iloc[0])
print("\nExample after cleaning:")
print(X_train_text_5[0])

## 4. TF-IDF representation and training-only balancing

TF-IDF is fitted on the original five-class training texts only. The vocabulary and IDF weights therefore do not use validation or test data. Random oversampling is applied after vectorization and only to training examples.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)

vectorizer_5 = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=1,
    max_features=50000,
    sublinear_tf=False,
)
X_train_5 = vectorizer_5.fit_transform(X_train_text_5)
X_val_5 = vectorizer_5.transform(X_val_text_5)
X_test_5 = vectorizer_5.transform(X_test_text_5)

ros_5 = RandomOverSampler(random_state=SEED)
X_train_5_bal, y_train_5_bal = ros_5.fit_resample(X_train_5, y_train_5)

print("TF-IDF vocabulary size:", len(vectorizer_5.vocabulary_))
print("Original training matrix:", X_train_5.shape)
print("Balanced training matrix:", X_train_5_bal.shape)

## 5. Naive Bayes baseline

In [ ]:
nb_5 = MultinomialNB()
nb_5.fit(X_train_5_bal, y_train_5_bal)
nb_pred_5 = nb_5.predict(X_test_5)

print("Five-class Naive Bayes")
print(f"Accuracy: {accuracy_score(y_test_5, nb_pred_5):.4f}")
print(f"Macro-F1: {f1_score(y_test_5, nb_pred_5, average='macro'):.4f}")
print(classification_report(
    y_test_5, nb_pred_5,
    labels=RETAINED_LABELS,
    target_names=RETAINED_LABELS,
    digits=3,
    zero_division=0,
))

## 6. Logistic Regression — main classical model

We select `C` from a small predefined grid using validation macro-F1. The test set is not used for hyperparameter selection.

In [ ]:
C_CANDIDATES = [0.5, 1.0, 2.0]
validation_rows_5 = []
models_5_by_c = {}

for c in C_CANDIDATES:
    candidate = LogisticRegression(C=c, max_iter=2000, random_state=SEED)
    candidate.fit(X_train_5_bal, y_train_5_bal)
    val_pred = candidate.predict(X_val_5)
    val_macro_f1 = f1_score(y_val_5, val_pred, average="macro")
    validation_rows_5.append({"C": c, "validation_macro_f1": val_macro_f1})
    models_5_by_c[c] = candidate

validation_results_5 = pd.DataFrame(validation_rows_5).sort_values(
    "validation_macro_f1", ascending=False
)
display(validation_results_5)
BEST_C_5 = float(validation_results_5.iloc[0]["C"])
lr_5 = models_5_by_c[BEST_C_5]
lr_pred_5 = lr_5.predict(X_test_5)

lr_accuracy_5 = accuracy_score(y_test_5, lr_pred_5)
lr_macro_f1_5 = f1_score(y_test_5, lr_pred_5, average="macro")
print(f"Selected C: {BEST_C_5}")
print(f"Test accuracy: {lr_accuracy_5:.4f}")
print(f"Test macro-F1: {lr_macro_f1_5:.4f}")

report_5 = classification_report(
    y_test_5, lr_pred_5,
    labels=RETAINED_LABELS,
    target_names=RETAINED_LABELS,
    digits=3,
    output_dict=True,
    zero_division=0,
)
display(pd.DataFrame(report_5).T.round(3))

In [ ]:
cm_5 = confusion_matrix(y_test_5, lr_pred_5, labels=RETAINED_LABELS)
cm_5_norm = confusion_matrix(y_test_5, lr_pred_5, labels=RETAINED_LABELS, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(cm_5, annot=True, fmt="d", cmap="Blues",
            xticklabels=RETAINED_LABELS, yticklabels=RETAINED_LABELS, ax=axes[0])
axes[0].set_title("Five-class LR — counts")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
sns.heatmap(cm_5_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=RETAINED_LABELS, yticklabels=RETAINED_LABELS, ax=axes[1])
axes[1].set_title("Five-class LR — row-normalized")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

## 7. Error analysis

Inspect the largest off-diagonal pair and show real held-out examples. Treat individual tweets as context-dependent examples, not definitive evidence that the dataset's label is wrong.

In [ ]:
cm_no_diag_5 = cm_5.copy()
np.fill_diagonal(cm_no_diag_5, 0)
worst_i, worst_j = np.unravel_index(np.argmax(cm_no_diag_5), cm_no_diag_5.shape)
print(f"Largest confusion: true={RETAINED_LABELS[worst_i]} -> predicted={RETAINED_LABELS[worst_j]} "
      f"({cm_5[worst_i, worst_j]} test examples)")

error_view = pd.DataFrame({
    "raw_text": test_5["text"].to_numpy(),
    "clean_text": X_test_text_5,
    "true_label": y_test_5,
    "predicted_label": lr_pred_5,
})
display(error_view[
    (error_view["true_label"] == RETAINED_LABELS[worst_i]) &
    (error_view["predicted_label"] == RETAINED_LABELS[worst_j])
].head(15))

## 8. Model interpretability: influential TF-IDF features

For each class, the highest positive coefficients indicate features that push the linear classifier toward that class relative to the other classes under its learned decision function. These are associations, not causal explanations.

In [ ]:
feature_names = np.asarray(vectorizer_5.get_feature_names_out())
TOP_N = 15
feature_rows = []
for class_idx, class_name in enumerate(lr_5.classes_):
    top_idx = np.argsort(lr_5.coef_[class_idx])[-TOP_N:][::-1]
    for rank, idx in enumerate(top_idx, start=1):
        feature_rows.append({
            "class": class_name,
            "rank": rank,
            "feature": feature_names[idx],
            "coefficient": lr_5.coef_[class_idx, idx],
        })
top_features_df = pd.DataFrame(feature_rows)
for class_name in lr_5.classes_:
    print(f"\nTop features for {class_name}")
    display(top_features_df[top_features_df["class"] == class_name][["rank", "feature", "coefficient"]].reset_index(drop=True))

## 9. Matched six-class vs five-class comparison

This section intentionally trains a six-class LR control again so Notebook 02 can reproduce the comparison on its own. Both model formulations use the same 70/15/15 split seed and the **same held-out rows whose true labels are in the five retained classes**. The six-class model can still predict `other_cyberbullying`; those predictions count as errors in this retained-class evaluation.

Each model tunes `C` on its own validation labels, then is evaluated on the same retained test rows. This compares practical end-to-end formulations; the five-class model is not expected to be directly equivalent to the six-class model.

In [ ]:
# Reproduce the six-class LR control with the same classical cleaner and split.
X_train_text_6_ctrl = train_df["text"].map(clean_tweet_classical).tolist()
X_val_text_6_ctrl = val_df["text"].map(clean_tweet_classical).tolist()
X_test_text_6_ctrl = test_df["text"].map(clean_tweet_classical).tolist()
y_train_6_ctrl = train_df["label"].to_numpy()
y_val_6_ctrl = val_df["label"].to_numpy()

tfidf_6_ctrl = TfidfVectorizer(
    ngram_range=(1, 1), min_df=1, max_features=50000, sublinear_tf=False
)
Xtr6 = tfidf_6_ctrl.fit_transform(X_train_text_6_ctrl)
Xv6 = tfidf_6_ctrl.transform(X_val_text_6_ctrl)
Xte6 = tfidf_6_ctrl.transform(X_test_text_6_ctrl)
Xtr6_bal, ytr6_bal = RandomOverSampler(random_state=SEED).fit_resample(Xtr6, y_train_6_ctrl)

control_scores = []
control_models = {}
for c in C_CANDIDATES:
    m = LogisticRegression(C=c, max_iter=2000, random_state=SEED)
    m.fit(Xtr6_bal, ytr6_bal)
    score = f1_score(y_val_6_ctrl, m.predict(Xv6), average="macro")
    control_scores.append({"C": c, "validation_macro_f1": score})
    control_models[c] = m
BEST_C_6_CTRL = float(pd.DataFrame(control_scores).sort_values(
    "validation_macro_f1", ascending=False
).iloc[0]["C"])
lr_6_ctrl = control_models[BEST_C_6_CTRL]

retained_mask = test_df["label"].isin(RETAINED_LABELS).to_numpy()
y_true_common = test_df.loc[retained_mask, "label"].to_numpy()
y_pred_6_common = lr_6_ctrl.predict(Xte6[retained_mask])
y_pred_5_common = lr_5.predict(X_test_5)

p6, r6, f6, s6 = precision_recall_fscore_support(
    y_true_common, y_pred_6_common, labels=RETAINED_LABELS, zero_division=0
)
p5, r5, f5, s5 = precision_recall_fscore_support(
    y_true_common, y_pred_5_common, labels=RETAINED_LABELS, zero_division=0
)

comparison_rows = [
    {
        "formulation": "Six-class LR, scored on retained-class test rows",
        "accuracy": accuracy_score(y_true_common, y_pred_6_common),
        "macro_f1_over_retained_classes": np.mean(f6),
        "predicted_other_on_retained_rows": int(np.sum(y_pred_6_common == OTHER_LABEL)),
    },
    {
        "formulation": "Five-class LR",
        "accuracy": accuracy_score(y_true_common, y_pred_5_common),
        "macro_f1_over_retained_classes": np.mean(f5),
        "predicted_other_on_retained_rows": 0,
    },
]
display(pd.DataFrame(comparison_rows).round(4))

class_comparison = pd.DataFrame({
    "class": RETAINED_LABELS,
    "f1_six_class_model": f6,
    "f1_five_class_model": f5,
    "support": s5,
})
display(class_comparison.round(3))
print("Shared retained test rows:", len(y_true_common))
print("Six-class control selected C:", BEST_C_6_CTRL)
print("Five-class model selected C:", BEST_C_5)

## 10. Final interpretation

Report accuracy, macro-F1, per-class metrics, the confusion matrix, and representative errors. Use the matched comparison above to describe how filtering the original problem to five labels changes performance on identical retained test examples.

Do not claim that performance improvement proves the removed class is invalid. The strongest scope rationale is that this project deliberately focuses on four identity-targeted categories plus non-bullying content; the diagnostics are supporting context.